# 09-phase10-channel-graph-foundations

**neuron Phase 10** — channel-as-node graph hidden layer foundations (사용자 vision 의 본질 axis).

핵심 가설:
1. **function preservation** — channel_full (adj=full) 가 plain Linear 와 forward 동치?
2. **adjacency 학습** — adj 파라미터에 gradient 흐름 + 학습된 per-edge importance?
3. **0-init 금지 + sweet spot 적용** — channel_uniform_small (adj ∈ [0.05, 0.15]) 시작이 plain 과 비교?
4. **post-training adj 분포** — Phase 5 의 implicit pruning 패턴이 edge-level 에서 재현?
5. **Phase 9 group 과의 비교** — channel-level granularity 가 group-level (Phase 9: 2.1378~2.1391) 보다 우위?

설계: 3-way sweep × 2 seed = 6 run, max_steps=1500.
- arch ∈ {plain, channel_full, channel_uniform_small}
- seed ∈ {42, 123}

데이터: TinyShakespeare (char-LM)
시드: [42, 123]
작성일: 2026-05-26
연관: Issue [#61](https://github.com/EinSofINTEREST/GraphLM/issues/61) / Phase 9 baseline PR [#60](https://github.com/EinSofINTEREST/GraphLM/pull/60) / [feedback_no_zero_init.md](feedback memory)

## 1. 환경

In [ ]:
import logging
import sys

import torch

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.graph_channel_demo import train_channel_graph_mlp
from graphlm.utils import repo_root

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)
print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)

## 2. 실험 설정

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase10"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123]
ARCHS = ["plain", "channel_full", "channel_uniform_small"]
EMB_DIM = 64
HIDDEN_DIM = 256  # channel granularity — group 무관, 어떤 dim 도 가능
N_GRAM = 4
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500

# Phase 9 baseline (PR #60 결과)
PHASE9_PLAIN_MEAN = 2.1378
PHASE9_GROUP_FULL_MEAN = 2.1391
PHASE9_GROUP_IDENTITY_MEAN = 2.2797  # 0-init vanishing 사례

print(f"SEEDS      = {SEEDS}")
print(f"ARCHS      = {ARCHS}")
print(f"HIDDEN_DIM = {HIDDEN_DIM} (channel granularity)")
print(f"MAX_STEPS  = {MAX_STEPS}")
print(f"전체 run   = {len(SEEDS) * len(ARCHS)}")

## 3. 데이터 로드

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
V = tokenizer.vocab_size
print(f"vocab_size : {V}")  # padding 불필요 — channel granularity 는 모든 dim 지원

## 4. sweep 학습

각 (seed, arch) 에 대해 1 run.
- plain: standard nn.Linear baseline
- channel_full: adj=full (function preservation, 'free' graph)
- channel_uniform_small: adj ∈ [0.05, 0.15] (Phase 2 sweet spot 패턴 적용, 0-init 회피)

In [ ]:
runs = {}
for seed in SEEDS:
    for arch in ARCHS:
        key = (seed, arch)
        print(f"--- seed={seed}, arch={arch} ---")
        runs[key] = train_channel_graph_mlp(
            dataset=dataset,
            vocab_size=V,
            seed=seed,
            arch=arch,
            emb_dim=EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            n_gram=N_GRAM,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            device=DEVICE,
        )
        print(f"  done: final_loss={runs[key]['final_loss']:.4f}")
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 5. 결과 표 — arch × seed + Phase 9 baseline 비교

In [ ]:
import statistics

print(f"{'arch':>26}  {'seed':>5}  {'final_loss':>11}")
print("-" * 50)
for arch in ARCHS:
    for seed in SEEDS:
        r = runs[(seed, arch)]
        print(f"{arch:>26}  {seed:>5}  {r['final_loss']:>11.4f}")

print()
print("=== arch 별 mean ===")
agg = {}
for arch in ARCHS:
    fls = [runs[(s, arch)]["final_loss"] for s in SEEDS]
    agg[arch] = dict(mean=statistics.mean(fls), range=max(fls) - min(fls))
    print(f"  {arch:>26}: mean={agg[arch]['mean']:.4f}, range={agg[arch]['range']:.4f}")

print()
print("=== Phase 9 (group-level) baseline 비교 ===")
print(f"  Phase 9 plain               : {PHASE9_PLAIN_MEAN:.4f}")
print(f"  Phase 9 group_full          : {PHASE9_GROUP_FULL_MEAN:.4f}")
print(f"  Phase 9 group_identity      : {PHASE9_GROUP_IDENTITY_MEAN:.4f}  (0-init vanishing 사례)")
print()
print(f"  Phase 10 plain (재현)        : {agg['plain']['mean']:.4f}")
print(f"  Phase 10 channel_full       : {agg['channel_full']['mean']:.4f}")
print(f"  Phase 10 channel_uniform    : {agg['channel_uniform_small']['mean']:.4f}")

## 6. 학습된 adj 분포 분석 — implicit pruning at edge-level?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# channel_* 의 fc1 adj 분포 vs init
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for col_i, arch in enumerate(["channel_full", "channel_uniform_small"]):
    r = runs[(SEEDS[0], arch)]
    if r["final_adj"] is None:
        continue
    for row_i, layer in enumerate(["fc1", "fc2"]):
        ax = axes[row_i, col_i]
        adj = r["final_adj"][layer].numpy().flatten()
        ax.hist(adj, bins=80, alpha=0.7, color="#1f77b4")
        ax.set_xlabel("adj value")
        ax.set_ylabel("count")
        ax.set_title(f"{arch} — {layer} adj distribution (n={len(adj)})")
        ax.axvline(0, color="red", linestyle="--", lw=0.8, alpha=0.5)
        ax.axvline(adj.mean(), color="green", linestyle=":", lw=1, label=f"mean={adj.mean():.3f}")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

fig.suptitle(f"Phase 10 — channel-level adj 분포 (seed={SEEDS[0]})", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "adj_distribution.png", dpi=120)
plt.show()

## 7. adj heatmap (fc1, channel × channel)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, arch in enumerate(["channel_full", "channel_uniform_small"]):
    r = runs[(SEEDS[0], arch)]
    if r["final_adj"] is None:
        continue
    adj = r["final_adj"]["fc1"].numpy()
    vmax = max(abs(adj).max(), 1e-6)
    im = axes[i].imshow(adj, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    axes[i].set_xlabel("input channel")
    axes[i].set_ylabel("output channel")
    axes[i].set_title(f"{arch} — fc1 adj (shape={adj.shape})")
    fig.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

fig.suptitle(f"Phase 10 — channel adj heatmap (seed={SEEDS[0]}, fc1)", fontsize=11)
fig.tight_layout()
fig.savefig(OUT_DIR / "adj_heatmaps.png", dpi=120)
plt.show()

## 8. loss curve 비교 (arch × mean ± σ across 2 seeds)

In [ ]:
window = 30
colors = {
    "plain": "#1f77b4",
    "channel_full": "#2ca02c",
    "channel_uniform_small": "#ff7f0e",
}

fig, ax = plt.subplots(figsize=(13, 5))
for arch in ARCHS:
    seed_curves = []
    for seed in SEEDS:
        losses = runs[(seed, arch)]["losses"]
        smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
        seed_curves.append(smoothed)
    arr = np.array(seed_curves)
    steps = np.arange(window - 1, window - 1 + arr.shape[1])
    mean = arr.mean(axis=0)
    std = arr.std(axis=0, ddof=1)
    ax.plot(steps, mean, color=colors[arch], lw=1.5, label=arch)
    ax.fill_between(steps, mean - std, mean + std, color=colors[arch], alpha=0.15)
ax.axhline(
    PHASE9_GROUP_FULL_MEAN,
    color="gray",
    linestyle=":",
    lw=1,
    alpha=0.7,
    label=f"Phase 9 group_full ({PHASE9_GROUP_FULL_MEAN})",
)
ax.set_xlabel("step")
ax.set_ylabel(f"loss (smoothed window={window})")
ax.set_title(f"Phase 10 — plain Linear vs ChannelGraphLinear (mean ± σ over {len(SEEDS)} seeds)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "loss_curves.png", dpi=120)
plt.show()

## 결과 요약 / Phase 11 권장 방향

확인 포인트:
- §5 channel_full vs plain — function preservation 입증 (~0)?
- §5 channel_uniform_small vs plain — 0-init 회피한 sweet spot 패턴이 잘 작동? (열위 ≤ 0.05 면 OK)
- §6 adj 분포 — 학습 후 spread? 일부 edge 가 자연 약화 (implicit pruning)?
- §7 heatmap — sparse/structured 패턴 emerge?
- §8 Phase 9 group_full (2.1391) 와의 비교 — channel granularity 가 group 보다 우위/동등?

**판정 시나리오**:
- **A. channel_full ≈ plain + uniform_small 도 비슷** ⭐ — function preservation + 0-init 회피 둘 다 입증, Phase 11 (hybrid) 진입
- **B. channel_full ≈ plain, channel_uniform_small 열위** — sweet spot 의 small init 이 fine-grained edge 에는 부족, 추가 sweep 필요
- **C. channel_uniform_small 우위** — implicit pruning at edge-level 이 실제 효과 입증 (sparsification 학습 motivation 강화)

**참고**:
- 아키텍처 구성 계획 (Notion): https://www.notion.so/36ce8b70b7aa818cbf1fe71687b449b8
- Phase 9 결과: https://www.notion.so/36ce8b70b7aa8100b0acf756686d2e9f